# Embedding and search (KAN-97)
Gözde Nur Özdemir · ELIZA · Sprint 2

**What the course asks for in Sprint 2:** *"the most basic vector database and retrieval over your documents"*, with the parameters in `config/rag.yaml`, and by 11/11: *"retrieval returns passages for a handful of test questions"*.

**What this notebook does:** it turns our 105 chunks into vectors, stores them, and returns the 5 best chunks for a question. Every choice says where it comes from.

| Step | What | Where the choice comes from |
|---|---|---|
| 1 | Load the settings and the chunks | `config/rag.yaml` (course rule) |
| 2 | Prepare the text for the two kinds of search | the meaning of the law (negations) |
| 3 | Frequent words: LLM judge + my check | instructor's feedback |
| 4 | Embed the chunks with `bge-base-en-v1.5` | design decision D10, model card |
| 5 | Store them in Chroma | design decision D11 |
| 6 | Keyword index (BM25) | design decision D12 |
| 7 | Search: dense (E1) and hybrid (E2) | design decision D12, RRF paper |
| 8 | A handful of test questions (11/11 check) | course halfway check |
| 9 | A picture of the chunks | my earlier retrieval project |

The measurement on the `val` questions (Recall@5, MRR, the instructor's RAG metrics, W&B) is the next task, KAN-99.

In [ ]:
%pip -q install sentence-transformers chromadb rank_bm25 pandas pyyaml requests scikit-learn umap-learn matplotlib
import csv, json, re
from collections import Counter
import numpy as np, pandas as pd, requests, yaml
pd.set_option("display.max_colwidth", None)
np.random.seed(42)

## 1. Load the settings and the chunks
All settings are in `config/rag.yaml`: the course asks for parameters there, not in the code. The chunks come from `src/chunking.ipynb`.

In [ ]:
CONFIG = yaml.safe_load(open("../config/rag.yaml"))
chunks = [json.loads(line) for line in open("../data/chunks.jsonl", encoding="utf-8")]
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}
print(len(chunks), "chunks")
CONFIG

## 2. Prepare the text for the two kinds of search
**Dense search (embedding)** reads the full `embed_text`: heading + the sentence before a point + the provision. We keep every word, because the model reads normal sentences and small words change a rule: *"shall **not** be considered to be high-risk"* means the opposite without "not".

**Keyword search (BM25)** reads the same text in lower case, split into words, without stopwords ("the", "of", "which"), which are in almost every chunk. We use the standard English list of scikit-learn, but **keep "not", "no" and "nor"**, for the same reason.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STOPWORDS = set(ENGLISH_STOP_WORDS) - {"not", "no", "nor"}
words = lambda text: re.findall(r"[a-z0-9]+", text.lower())

example = next(c for c in chunks if c["chunk_id"] == "ART6-3-1")
print("TEXT: ", example["text"][:150], "...")
print("BM25: ", [w for w in words(example["text"]) if w not in STOPWORDS][:18], "...")
print(sum(any(w in {"not", "no", "nor"} for w in words(c["text"])) for c in chunks), "of 105 chunks contain a negation")

**Comment:** _(write 1–2 sentences after running)_

## 3. Frequent words: LLM judge + my check
Some words that are not normal stopwords are still in many chunks ("shall", "referred", "high-risk"). The instructor's advice: do **not** remove words only because they are frequent (for example with a TF-IDF cut), because a frequent word can carry the meaning of a question; "high-risk" is in half of the chunks, but almost every question is about it.

So an LLM reads each frequent word with three examples from our chunks and says **KEEP** or **IGNORE** with a reason; then I check every label. Only the words marked IGNORE after my check are left out of BM25.

**Why Qwen3-4B as the judge**
1. It is from a **different model family** than our answer model (Llama-3.1-8B, D9); the instructor asked for a different model as judge.
2. In our model selection spike it answered correctly with thinking turned off.
3. It runs on a laptop with Ollama, which the project already uses (D9). Temperature 0, so the labels repeat. Needs `ollama pull qwen3:4b`.

In [ ]:
in_chunks = Counter(w for c in chunks for w in set(words(c["text"])))
frequent = [w for w, n in in_chunks.most_common() if n / len(chunks) >= 0.2 and w not in STOPWORDS]

PROMPT = """This word appears in many EU AI Act provisions. Examples:
{examples}
Should a keyword search IGNORE it (common wording that does not help to find the right provision)
or KEEP it (a question can depend on it)? If in doubt, KEEP.
Answer in one line: IGNORE or KEEP, then a reason of at most 12 words.
WORD: {word}"""

def ask(prompt):
    """One answer from the judge model (Ollama, temperature 0, thinking off)."""
    r = requests.post(CONFIG["judge"]["url"], json={"model": CONFIG["judge"]["model"], "think": False, "stream": False,
                      "options": {"temperature": 0}, "messages": [{"role": "user", "content": prompt}]})
    return r.json()["message"]["content"].strip()

def examples(word):
    return "\n".join(f'- {c["ref"]}: {c["text"][:150]}...' for c in chunks if word in words(c["text"]))

judge = pd.DataFrame([{"word": w, "% of chunks": round(100 * in_chunks[w] / len(chunks)),
                       "LLM": ask(PROMPT.format(examples="\n".join(examples(w).split("\n")[:3]), word=w))} for w in frequent])
judge

**My check.** I read every label. Where I disagree, I write my decision here, e.g. `{"risk": "KEEP"}`.

In [ ]:
MY_CHANGES = {}
judge["final"] = [MY_CHANGES.get(w, "IGNORE" if "IGNORE" in a.upper() else "KEEP") for w, a in zip(judge["word"], judge["LLM"])]
IGNORE = set(judge.loc[judge["final"] == "IGNORE", "word"])
print("Changed by me:", MY_CHANGES or "none", "| BM25 also ignores:", sorted(IGNORE))

**Comment:** _(write 1–2 sentences after running)_

## 4. Embed the chunks
`BAAI/bge-base-en-v1.5` (D10, chosen in the model selection spike) turns each text into 768 numbers; texts with a similar meaning get similar numbers. The vectors are normalised, so cosine similarity is a dot product. As its model card recommends, questions get a short instruction in front (`query_instruction` in the config); chunks do not.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(CONFIG["embedding"]["model"], device="cpu")
embed = lambda texts: model.encode(texts, normalize_embeddings=True)

vectors = embed([c["embed_text"] for c in chunks])
print("vectors:", vectors.shape)

i = {c["chunk_id"]: n for n, c in enumerate(chunks)}
print("ART6-3-a vs ART6-3-b (two points of the same rule):", round(float(vectors[i["ART6-3-a"]] @ vectors[i["ART6-3-b"]]), 2))
print("ART6-3-a vs ART113-3-c-i (a different topic):      ", round(float(vectors[i["ART6-3-a"]] @ vectors[i["ART113-3-c-i"]]), 2))

Two points of the same rule should be closer (higher number) than two different topics.

**Comment:** _(write 1–2 sentences after running)_

## 5. Store them in Chroma
Chroma (D11) keeps the vectors, the texts and the metadata, and returns the closest chunks with cosine distance. It is saved in `data/vector-store/`; it is not in Git, because this notebook rebuilds it in seconds.

In [ ]:
import chromadb
store = chromadb.PersistentClient(path=CONFIG["vector_store"]["path"])
if "chunks" in [c.name for c in store.list_collections()]:
    store.delete_collection("chunks")
col = store.create_collection("chunks", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["chunk_id"] for c in chunks], embeddings=vectors.tolist(), documents=[c["text"] for c in chunks],
        metadatas=[{"ref": c["ref"], "version_date": c["version_date"]} for c in chunks])
print(col.count(), "chunks stored")

## 6. Keyword index (BM25)
BM25 (D12) gives a high score to chunks that share the question's words, especially rare ones. It finds exact terms such as "Annex III" or "Article 113" that the embedding can miss.

In [ ]:
from rank_bm25 import BM25Okapi
keep = lambda text: [w for w in words(text) if w not in STOPWORDS | IGNORE]
bm25 = BM25Okapi([keep(c["embed_text"]) for c in chunks])
print("BM25 index over", len(chunks), "chunks")

## 7. Search: dense (E1) and hybrid (E2)
- **Dense (E1, the baseline):** the closest vectors in Chroma.
- **Hybrid (E2):** the dense list and the BM25 list joined with Reciprocal Rank Fusion: each chunk gets `1 / (60 + rank)` from each list, and the two are added. 60 is the value from the RRF paper (Cormack et al., 2009).

Both return the top 5 (`top_k`); which one we keep, and whether 5 is right, is measured on the `val` questions in KAN-99.

In [ ]:
R = CONFIG["retrieval"]

def dense(question):
    q = embed([CONFIG["embedding"]["query_instruction"] + question])[0]
    return col.query(query_embeddings=[q.tolist()], n_results=R["candidates"])["ids"][0]

def keyword(question):
    return [chunks[n]["chunk_id"] for n in np.argsort(-bm25.get_scores(keep(question)))[:R["candidates"]]]

def hybrid(question):
    rrf = Counter()
    for ranked in (dense(question), keyword(question)):
        for rank, cid in enumerate(ranked, 1):
            rrf[cid] += 1 / (R["rrf_k"] + rank)
    return [cid for cid, _ in rrf.most_common(R["top_k"])]

[ref_of[c] for c in hybrid("Is an AI tool that grades students' essays high-risk?")]

**Comment:** _(write 1–2 sentences after running)_

## 8. A handful of test questions (11/11 check)
Five questions from `docs/eval/eval-set-v1.csv`, from Yeşim's (BM), Yaren's (EM) and my (B) questions. Next to the top 5 chunks of dense and hybrid search is the **gold source** the team wrote; ✓ = a gold provision.

In [ ]:
questions = {r["id"]: r for r in csv.DictReader(open("../docs/eval/eval-set-v1.csv", encoding="utf-8-sig"))}

def show(ids, gold):
    parts = [g.strip() for g in gold.split(";")]
    return "\n".join(("✓ " if any(ref_of[c].startswith(g) or g.startswith(ref_of[c]) for g in parts) else "· ") + ref_of[c] for c in ids)

rows = [{"id": q, "question": questions[q]["question"], "dense (E1)": show(dense(questions[q]["question"])[:5], questions[q]["gold_source"]),
         "hybrid (E2)": show(hybrid(questions[q]["question"]), questions[q]["gold_source"]), "gold source": questions[q]["gold_source"]}
        for q in ["BM-01", "BM-07", "EM-09", "B-03", "B-04"]]
pd.DataFrame(rows).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

**Comment:** _(write 1–2 sentences after running)_

## 9. A picture of the chunks
As in my earlier retrieval project, UMAP draws the 768 numbers of each chunk on a page. Chunks with a similar meaning should be close; each colour is one area of our provision list.

In [ ]:
import umap, matplotlib.pyplot as plt
xy = umap.UMAP(n_neighbors=10, random_state=42).fit_transform(vectors)
for area in sorted({c["area"] for c in chunks}):
    pts = xy[[c["area"] == area for c in chunks]]
    plt.scatter(pts[:, 0], pts[:, 1], s=25, label=area)
plt.legend(fontsize=7); plt.title("105 chunks, coloured by area"); plt.axis("off"); plt.show()

**Comment:** _(write 1–2 sentences after running)_

## Next: KAN-99
- Split the questions into `val` and `test` (KAN-53, 11/11).
- On every `val` question: Recall@5 and MRR (instructor's slides: *"compute Recall@k / MRR on gold IDs"*), dense (E1) vs hybrid (E2), and top-k 3, 5 or 8.
- The instructor's RAG metrics: Context Entities Recall (*"for fact-heavy text such as regulations"*) and, with the generated answers, Faithfulness.
- Everything logged in W&B; the baseline experiment card is KAN-100.